# Bài 54: Segment Anything (Segment Anything)

Phân đoạn thực thể ở Bài 43 huấn luyện một mạng đưa ra mặt nạ của *mọi* thực thể cùng lúc, rồi gỡ rối chúng bằng bỏ phiếu tâm. **Segment Anything (SAM)** (<a href="../references.html#kirillov-2023-sam">Kirillov et al., 2023</a>) đi theo một cách tiếp cận khác: phân đoạn theo yêu cầu. Cho mô hình một *lời nhắc (prompt)* &mdash; một điểm, một hộp, một nét vẽ nguệch ngoạc &mdash; và nó trả về mặt nạ của đúng một đối tượng mà lời nhắc đó chỉ tới. Bài này xây dựng ý tưởng cốt lõi của SAM: đưa lời nhắc vào như một tín hiệu không gian tường minh bên cạnh ảnh, và huấn luyện mạng tạo ra chính xác mặt nạ của đối tượng được nhắc, không gì khác.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Các cảnh với một lời nhắc điểm (Scenes with a point prompt)
Dùng lại các cảnh hình tròn chồng lấn ở Bài 43. Thay vì dự đoán mọi thực thể cùng lúc, lấy mẫu một điểm *bên trong* một trong các thực thể ("lời nhắc") và huấn luyện mô hình chỉ đưa ra mặt nạ của thực thể đó.

In [ ]:
SIZE = 32

def make_scene(rng, size=SIZE, n_circles=2, r=5):
    scene = np.zeros((size, size), dtype=np.float32)
    inst_mask = np.zeros((size, size), dtype=np.int64)
    centers = []
    for k in range(n_circles):
        if k == 0:
            cx, cy = rng.integers(r + 8, size - r - 8), rng.integers(r + 8, size - r - 8)
        else:
            px, py = centers[-1]
            angle = rng.uniform(0, 2 * np.pi)
            dist = rng.uniform(6, 9)  # overlapping but centers still separable
            cx = int(np.clip(px + dist * np.cos(angle), r, size - r - 1))
            cy = int(np.clip(py + dist * np.sin(angle), r, size - r - 1))
        centers.append((cx, cy))
        yy, xx = np.mgrid[0:size, 0:size]
        m = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
        scene[m] = 1.0
        inst_mask[m] = k + 1
    scene = np.clip(scene + rng.normal(0, 0.05, scene.shape), 0, 1).astype(np.float32)
    return scene, inst_mask, centers

def sample_point_prompt(inst_mask, instance_id, rng):
    ys, xs = np.where(inst_mask == instance_id)
    i = rng.integers(len(xs))
    return xs[i], ys[i]

rng = np.random.default_rng(17)
N = 400
scenes, inst_masks, all_centers, prompts, prompt_targets = [], [], [], [], []
for _ in range(N):
    s, im, c = make_scene(rng)
    scenes.append(s); inst_masks.append(im); all_centers.append(c)
    inst_id = rng.integers(1, len(c) + 1)  # pick a random instance to prompt for
    px, py = sample_point_prompt(im, inst_id, rng)
    prompts.append((px, py))
    prompt_targets.append((im == inst_id).astype(np.float32))

scenes = np.array(scenes, dtype=np.float32)
prompt_targets = np.array(prompt_targets, dtype=np.float32)
prompts_arr = np.array(prompts, dtype=np.float32)

split = int(0.85 * N)
Xtr, Ptr, Mtr = scenes[:split], prompts_arr[:split], prompt_targets[:split]
Xte, Pte, Mte = scenes[split:], prompts_arr[split:], prompt_targets[split:]
inst_te, centers_te = inst_masks[split:], all_centers[split:]

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for i in range(4):
    axes[i].imshow(scenes[i], cmap='gray')
    axes[i].scatter([prompts[i][0]], [prompts[i][1]], c='red', marker='*', s=100)
    axes[i].axis('off')
plt.suptitle('Scenes with their point prompt (red star)')
plt.show()

## Đưa lời nhắc vào như một tín hiệu không gian (Injecting the prompt as a spatial signal)
Kiến trúc của SAM gồm ba thành phần: một **bộ mã hóa ảnh** (một mạng xương sống nặng, tốn kém khi chạy, được tính một lần cho mỗi ảnh), một **bộ mã hóa lời nhắc** gọn nhẹ (biến một điểm/hộp/mặt nạ thành một vector nhúng), và một **bộ giải mã mặt nạ** nhanh kết hợp hai thứ đó. Lựa chọn thiết kế quan trọng nhất cho một lời nhắc điểm: mã hóa *vị trí* của điểm về mặt không gian, chứ không phải như một vector trừu tượng được phát đều lên toàn bộ ảnh. Kết xuất lời nhắc thành một đốm Gauss trên kênh riêng của nó, nối với ảnh, cho mỗi phép tích chập một tín hiệu trực tiếp, có nhận biết vị trí.

In [ ]:
def point_prompt_map(points, size=SIZE):
    B = points.shape[0]
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing='ij')
    yy = yy.float().unsqueeze(0).expand(B, -1, -1)
    xx = xx.float().unsqueeze(0).expand(B, -1, -1)
    px, py = points[:, 0].view(B, 1, 1), points[:, 1].view(B, 1, 1)
    d2 = (xx - px) ** 2 + (yy - py) ** 2
    return torch.exp(-d2 / (2 * 2.0 ** 2))  # a Gaussian blob centered at the prompt point

class TinySAM(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.ReLU(),  # image channel + prompt channel
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1),
        )

    def forward(self, img, point):
        prompt_map = point_prompt_map(point).unsqueeze(1)
        return self.net(torch.cat([img, prompt_map], dim=1)).squeeze(1)

torch.manual_seed(0)
model = TinySAM()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); Pt = torch.tensor(Ptr); Mt = torch.tensor(Mtr)
for _ in range(300):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(Xt, Pt), Mt)
    loss.backward()
    opt.step()

with torch.no_grad():
    logits_te = model(torch.tensor(Xte).unsqueeze(1), torch.tensor(Pte))
    preds = (logits_te > 0).numpy().astype(bool)

def iou(a, b):
    inter, union = (a & b).sum(), (a | b).sum()
    return inter / union if union > 0 else float('nan')

ious = [iou(preds[i], Mte[i].astype(bool)) for i in range(len(Xte))]
print(f'point-prompted mask IoU: mean = {np.mean(ious):.3f}')

## Phép thử thực sự: lời nhắc có thực sự chọn được một thực thể không? (The real test: does the prompt actually select an instance?)
Lấy một cảnh kiểm tra có hai hình tròn chồng lấn, và truy vấn nó hai lần với hai lời nhắc điểm khác nhau — mỗi lời nhắc bên trong một hình tròn. Một mô hình thực sự được điều khiển bởi lời nhắc phải trả về hai mặt nạ *khác nhau*; một mô hình âm thầm bỏ qua lời nhắc và chỉ làm phân đoạn ngữ nghĩa (Bài 42) sẽ trả về cùng một mặt nạ "toàn bộ điểm ảnh hình tròn" cả hai lần.

In [ ]:
idx = 0
scene0 = Xte[idx]
(c1x, c1y), (c2x, c2y) = centers_te[idx]

with torch.no_grad():
    logit_a = model(torch.tensor(scene0[None, None]), torch.tensor([[c1x, c1y]], dtype=torch.float32))
    logit_b = model(torch.tensor(scene0[None, None]), torch.tensor([[c2x, c2y]], dtype=torch.float32))
mask_a, mask_b = (logit_a[0] > 0).numpy(), (logit_b[0] > 0).numpy()
overlap = (mask_a & mask_b).sum() / max((mask_a | mask_b).sum(), 1)

print(f'mask A size: {mask_a.sum()} px, mask B size: {mask_b.sum()} px')
print(f'overlap (IoU) between the two prompted masks: {overlap:.3f}  (low = genuinely different instances)')

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(scene0, cmap='gray')
axes[0].scatter([c1x, c2x], [c1y, c2y], c=['red', 'cyan'], marker='*', s=100)
axes[0].set_title('scene, both prompts'); axes[0].axis('off')
axes[1].imshow(mask_a, cmap='gray')
axes[1].scatter([c1x], [c1y], c='red', marker='*', s=100)
axes[1].set_title('mask for red prompt'); axes[1].axis('off')
axes[2].imshow(mask_b, cmap='gray')
axes[2].scatter([c2x], [c2y], c='cyan', marker='*', s=100)
axes[2].set_title('mask for cyan prompt'); axes[2].axis('off')
plt.show()

Hai lời nhắc trên cùng một ảnh không thay đổi tạo ra hai mặt nạ khác biệt rõ ràng, mỗi mặt nạ bám sát hình tròn mà lời nhắc thực sự rơi vào — mô hình thực sự đang điều kiện hóa theo lời nhắc, không chỉ chạy lại phân đoạn ngữ nghĩa và bỏ qua nó.

Điều này định hình lại hoàn toàn bài toán phân đoạn thực thể: Bài 43 cần một bước phân cụm chuyên biệt (bỏ phiếu, rồi gán) để khôi phục các thực thể từ một lượt xuôi duy nhất trên toàn ảnh. SAM né tránh điều đó bằng cách biến "thực thể nào" thành một *đầu vào* thay vì thứ cần suy ra — sự mơ hồ về ý nghĩa của một cú nhấp chuột được giải quyết bởi con người (hoặc một bộ phát hiện phía trước, Bài 41) cung cấp điểm, chứ không phải bởi mô hình phân đoạn đoán. Đây cũng là điều khiến tuyên bố "phân đoạn bất cứ thứ gì" của SAM khả thi: nó chưa bao giờ được huấn luyện trên các danh mục đối tượng cụ thể nào, chỉ trên nhiệm vụ tổng quát "đây là một điểm, đây là ranh giới đối tượng của nó" trên một tập dữ liệu khổng lồ, đa dạng (SA-1B, hơn 1 tỉ mặt nạ) — một kỹ năng không phân biệt danh mục có thể chuyển giao sang các loại đối tượng chưa từng thấy trong khi huấn luyện, giống như phân loại zero-shot ở Bài 52 đã chuyển giao sang các tên lớp chưa thấy.

### Bài tập (Exercises)
1. Đổi cách lấy mẫu lời nhắc trong vòng lặp huấn luyện để luôn chọn một điểm gần *cạnh* của thực thể thay vì bất kỳ đâu bên trong nó (`sample_point_prompt` hiện lấy mẫu đều). IoU với lời nhắc ở cạnh có thấp hơn IoU với lời nhắc ở bên trong không, và tại sao một mạng lại thấy các điểm ở cạnh mơ hồ hơn?
2. SAM trong thực tế hỗ trợ lời nhắc hộp cũng như lời nhắc điểm, và thường đưa ra *vài* mặt nạ ứng viên cho mỗi lời nhắc (vì một điểm đơn lẻ có thể mơ hồ — một điểm trên áo sơ mi có thể nghĩa là "cái áo" hoặc "cả người"). Hãy phác thảo cách bạn thay đổi lớp cuối của `TinySAM` để đưa ra 3 mặt nạ ứng viên thay vì 1, và bạn cần thêm tín hiệu huấn luyện nào để biết nên ưu tiên mặt nạ nào trong 3 mặt nạ đó.
3. Tăng `n_circles` từ 2 lên 4 trong `make_scene`, giữ mọi thứ khác như cũ, và chạy lại phép thử hai-lời-nhắc-khác-nhau trên một cảnh 4 hình tròn. Mô hình có còn tách gọn hai hình tròn cụ thể theo lời nhắc của chúng không, hay sự lộn xộn nhiều hơn làm suy giảm khả năng chọn lọc theo lời nhắc?